# E4 EN chạy lại trên corpus cùng số token (T13 phần 2)

Repo: https://github.com/KienNguyenDev2711/Hyena-Attention-Study

`E1_en` đã được thay bằng bản cắt đúng 38.250.964 token (phần 1), nhưng 6 run E4 EN
(`logspace`, `corpus`) vẫn train trên corpus cũ — mà dòng `uniform` của Bảng 7 lại
chính là `E1_en_HHHH`. Không chạy lại thì Bảng 4 và Bảng 7 vênh nhau cho cùng một
nhãn. Ở đây chạy lại 6 run E4 EN trên đúng corpus matched (cùng cache 110k bài,
cắt cùng đích); dòng `uniform` sau đó lấy từ E1_en matched.

Cấu hình giữ như E4 gốc: HHHH, nhánh corpus dùng alpha đo sẵn trong
`results/alpha_en_bpe500.json`. 6 run = logspace/corpus × seed 0–2, tốn ~12 phút
token hoá (phiên mới nên cache phải dựng lại) cộng ~1 giờ GPU T4, kết quả gói
trong `E4_en_matched_results.zip`.

Nhớ bật GPU trước: Runtime → Change runtime type → T4.


## 1 · Nạp mã nguồn

In [ ]:
REPO_URL = "https://github.com/KienNguyenDev2711/Hyena-Attention-Study.git"
WORK     = "/content/Hyena-Attention-Study"

import os, shutil, subprocess, sys

os.chdir("/content")             # thoát khỏi WORK trước khi xoá: rmtree thư mục
                                 # đang đứng làm git chết 128 "unable to read cwd"
if os.path.isdir(WORK):
    shutil.rmtree(WORK)          # chạy lại từ đầu -> luôn lấy bản mới nhất
subprocess.run(["git", "clone", "--depth", "1", "-q", REPO_URL, WORK], check=True)
os.chdir(WORK)
sys.path.insert(0, WORK)

print("Đã clone vào", WORK)


## 2 · Cài thư viện và kiểm tra GPU

In [ ]:
!pip install -q datasets tokenizers

import torch, datasets

print("torch", torch.__version__, "· datasets", datasets.__version__)
if torch.cuda.is_available():
    p = torch.cuda.get_device_properties(0)
    print(f"GPU: {p.name} · {p.total_memory/2**30:.1f} GB")
else:
    print("\n" + "!" * 70)
    print("!! CHƯA BẬT GPU — Runtime -> Change runtime type -> GPU (T4), rồi chạy lại")
    print("!" * 70)


## 3 · Kiểm trước khi đốt GPU

Ba bước, bước nào fail thì dừng: bản GitHub phải chứa E1_en matched (tức phần 1
đã tích hợp và push) cùng file alpha EN; test đường ống chạy CPU khoảng một phút;
dựng cache và xem tập train cắt ra đúng 38.250.964 chưa (~12 phút).


In [ ]:
import json

d = json.load(open("results/E1_en_HHHH_s0.json"))
assert d["corpus"]["n_tokens_train"] == 38_250_964, (
    "results/E1_en_HHHH_s0.json trên GitHub chưa phải bản matched — "
    "push phần tích hợp T13 phần 1 lên main rồi chạy lại từ ô clone."
)
alpha = json.load(open("results/alpha_en_bpe500.json"))
assert len(alpha["alpha"]) == 256, "file alpha EN thiếu hoặc sai số kênh"
print("OK: E1_en matched đã có trên repo, alpha EN sẵn sàng.")


In [ ]:
!python tests/test_pipeline.py


In [ ]:
from hyena_study.data import cached_token_stream

train, val, test, tok, stats = cached_token_stream(
    lang="en", tokenizer="syllable", vocab_size=16000, n_docs=110000,
    data_seed=0, max_tokens=38250964, cache_root="data_cache",
)
print(f"train={len(train):,} val={len(val):,} test={len(test):,}")
assert len(train) == 38250964, f"train={len(train):,} != 38.250.964"
print("OK: cache đã dựng, tập train cắt đúng 38.250.964 token.")


## 4 · Sáu lần chạy

Ba run `logspace` rồi ba run `corpus`, chạy tuần tự từng ô, ô nào lỗi chạy lại
riêng ô đó. Mỗi run để ý dòng `token: train=38,250,964`; riêng nhánh corpus phải
in `dùng alpha từ corpus: results/alpha_en_bpe500.json (256 kênh...)`.


In [ ]:
!python -m hyena_study.train --layers HHHH --lang en --tokenizer syllable \
        --n_docs 110000 --token_budget 50000000 --max_train_tokens 38250964 \
        --decay_mode logspace \
        --seed 0 --run_name E4_logspace_en_s0 --out_dir results_t13b


In [ ]:
!python -m hyena_study.train --layers HHHH --lang en --tokenizer syllable \
        --n_docs 110000 --token_budget 50000000 --max_train_tokens 38250964 \
        --decay_mode logspace \
        --seed 1 --run_name E4_logspace_en_s1 --out_dir results_t13b


In [ ]:
!python -m hyena_study.train --layers HHHH --lang en --tokenizer syllable \
        --n_docs 110000 --token_budget 50000000 --max_train_tokens 38250964 \
        --decay_mode logspace \
        --seed 2 --run_name E4_logspace_en_s2 --out_dir results_t13b


In [ ]:
!python -m hyena_study.train --layers HHHH --lang en --tokenizer syllable \
        --n_docs 110000 --token_budget 50000000 --max_train_tokens 38250964 \
        --decay_mode corpus --alpha_file results/alpha_en_bpe500.json \
        --seed 0 --run_name E4_corpus_en_s0 --out_dir results_t13b


In [ ]:
!python -m hyena_study.train --layers HHHH --lang en --tokenizer syllable \
        --n_docs 110000 --token_budget 50000000 --max_train_tokens 38250964 \
        --decay_mode corpus --alpha_file results/alpha_en_bpe500.json \
        --seed 1 --run_name E4_corpus_en_s1 --out_dir results_t13b


In [ ]:
!python -m hyena_study.train --layers HHHH --lang en --tokenizer syllable \
        --n_docs 110000 --token_budget 50000000 --max_train_tokens 38250964 \
        --decay_mode corpus --alpha_file results/alpha_en_bpe500.json \
        --seed 2 --run_name E4_corpus_en_s2 --out_dir results_t13b


## 5 · Kiểm và đóng gói

Mỗi file JSON phải ghi `corpus.n_tokens_train == 38.250.964` và `decay_mode`
khớp với `alpha_file`.


In [ ]:
import json, glob

files = sorted(glob.glob("results_t13b/E4_*_en_s*.json"))
assert len(files) == 6, f"kỳ vọng 6 file JSON, thấy {len(files)}: {files}"
print(f"{'run':22s} {'decay':>9s} {'train tokens':>14s} {'test PPL':>9s}")
for f in files:
    d = json.load(open(f))
    n = d["corpus"]["n_tokens_train"]
    assert n == 38250964, f"{f}: n_tokens_train={n} != 38250964"
    mode = d["config"]["decay_mode"]
    assert (mode == "corpus") == bool(d["config"]["alpha_file"]), f"{f}: alpha_file lệch decay_mode"
    print(f"{d['run_name']:22s} {mode:>9s} {n:>14,d} {d['test_ppl']:>9.3f}")
print("\nOK: cả 6 run E4 EN đều trên corpus matched.")


In [ ]:
!zip -qr E4_en_matched_results.zip results_t13b
try:
    from google.colab import files
    files.download("E4_en_matched_results.zip")
except Exception as e:
    print("Không tự tải được (", e, ")")
    print("-> Tải tay: panel Files bên trái -> E4_en_matched_results.zip")
